In [1]:
import torch
from transformers import AutoConfig, AutoModelForSequenceClassification, AutoTokenizer, TrainingArguments, Trainer, DataCollatorWithPadding
from Reader import obtain_combined_dataset
import numpy as np
from sklearn.metrics import f1_score, recall_score, precision_score

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    device = torch.device('cuda')
print("Current Device:", torch.cuda.current_device(), torch.cuda.get_device_name(torch.cuda.current_device()))

CUDA available: True
Current Device: 0 NVIDIA GeForce RTX 4070 Ti


In [2]:
datasets, label_list, label2id, id2label = obtain_combined_dataset(["TempEval3", "MAVEN", "TBDense"], "TempRel")

In [3]:
model_name = 'roberta-base'
tokenizer = AutoTokenizer.from_pretrained("roberta-base", use_fast=True, add_prefix_space=True)
config = AutoConfig.from_pretrained("roberta-base", num_labels=len(label_list), label2id=label2id, id2label=id2label)
model = AutoModelForSequenceClassification.from_pretrained(model_name, config=config)

seps = ["<e1>", "</e1>", "<e2>", "</e2>", "<e>", "</e>", "<timex", "</timex>", "TIMEVAL=", "TYPE=DATE>", "TYPE=TIME>", "TYPE=DURATION>", "TYPE=SET>", "TYPE=UNKOWN>"]
num_added = tokenizer.add_special_tokens({"additional_special_tokens": seps})
model.resize_token_embeddings(len(tokenizer))

d:\GeoTKG\venv\Lib\site-packages\huggingface_hub\file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of the model checkpoint at roberta-base were not used when initializing RobertaForSequenceClassification: ['lm_head.layer_norm.weight', 'lm_head.layer_norm.bias', 'lm_head.dense.bias', 'lm_head.bias', 'lm_head.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassifica

Embedding(50279, 768)

In [4]:
def compute_metrics(eval_prediction):
    predictions, ids = eval_prediction
    predictions = np.argmax(predictions, axis=-1)
    true_labels = [id2label[id] for id in ids]
    true_preds = [id2label[pred] for pred in predictions]
    return {
        "precision": precision_score(true_labels, true_preds, average="micro"),
        "recall": recall_score(true_labels, true_preds, average="micro"),
        "f1": f1_score(true_labels, true_preds, average="micro"),
    }

def encode_labels(example):
        return {"labels": label2id[example["label"]]}

def tokenize_batch(batch):
    # Let the collator pad/tensorize; just return lists
    return tokenizer(
        batch["tokens"],
        truncation=True,
        is_split_into_words=True
    )

In [5]:
# Tokenize + label encode
datasets = datasets.map(tokenize_batch, batched=True, remove_columns=[c for c in datasets["train"].column_names if c not in ["input_ids","attention_mask","label","tokens"]])
datasets = datasets.map(encode_labels)

# If a 'label' column still exists, keep only 'labels'
if "label" in datasets["train"].column_names:
    datasets = datasets.remove_columns("label")

# datasets["train"].to_json(".\\cleandata\\combined\\TempRel\\train.json")
# datasets["test"].to_json(".\\cleandata\\combined\\TempRel\\test.json")
# datasets["eval"].to_json(".\\cleandata\\combined\\TempRel\\eval.json")

Map:   0%|          | 0/887109 [00:00<?, ? examples/s]

Map:   0%|          | 0/887109 [00:00<?, ? examples/s]

In [6]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer, return_tensors="pt")

In [7]:
datasets

DatasetDict({
    test: Dataset({
        features: ['tokens', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 246420
    })
    train: Dataset({
        features: ['tokens', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 887109
    })
    eval: Dataset({
        features: ['tokens', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 98568
    })
})

In [9]:
training_args = TrainingArguments(
    output_dir="./results/TempRelTest",
    run_name="./results/TempRelTest",
    evaluation_strategy="steps",
    save_strategy="steps",
    logging_dir="./logs/TempRelTest",
    logging_steps=500,
    save_steps=10000,
    eval_steps=10000,
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=5e-5,
    metric_for_best_model="f1",
    load_best_model_at_end=True,
    report_to=["tensorboard"],
)

In [10]:
temprel_trainer = Trainer(
    model=model,
    args=training_args,
    tokenizer=tokenizer,
    data_collator=data_collator,
    train_dataset=datasets["train"],
    eval_dataset=datasets["eval"],
    compute_metrics=compute_metrics,
)

In [11]:
hist = temprel_trainer.train()

The following columns in the training set don't have a corresponding argument in `RobertaForSequenceClassification.forward` and have been ignored: tokens. If tokens are not expected by `RobertaForSequenceClassification.forward`,  you can safely ignore this message.
d:\GeoTKG\venv\Lib\site-packages\transformers\optimization.py:306: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(
***** Running training *****
  Num examples = 887109
  Num Epochs = 3
  Instantaneous batch size per device = 16
  Total train batch size (w. parallel, distributed & accumulation) = 16
  Gradient Accumulation steps = 1
  Total optimization steps = 166335
  Number of trainable parameters = 124662536


  0%|          | 0/166335 [00:00<?, ?it/s]

You're using a RobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


{'loss': 0.6911, 'learning_rate': 4.984970090480056e-05, 'epoch': 0.01}
{'loss': 0.6416, 'learning_rate': 4.969940180960111e-05, 'epoch': 0.02}
{'loss': 0.6601, 'learning_rate': 4.9549102714401664e-05, 'epoch': 0.03}
{'loss': 0.6539, 'learning_rate': 4.939880361920221e-05, 'epoch': 0.04}
{'loss': 0.7488, 'learning_rate': 4.924850452400277e-05, 'epoch': 0.05}
{'loss': 0.7003, 'learning_rate': 4.9098205428803325e-05, 'epoch': 0.05}
{'loss': 0.6463, 'learning_rate': 4.8947906333603875e-05, 'epoch': 0.06}
{'loss': 0.7971, 'learning_rate': 4.8797607238404424e-05, 'epoch': 0.07}
{'loss': 0.7874, 'learning_rate': 4.864730814320498e-05, 'epoch': 0.08}
{'loss': 0.788, 'learning_rate': 4.849700904800553e-05, 'epoch': 0.09}
{'loss': 0.7106, 'learning_rate': 4.8346709952806085e-05, 'epoch': 0.1}
{'loss': 0.641, 'learning_rate': 4.819641085760664e-05, 'epoch': 0.11}
{'loss': 0.6624, 'learning_rate': 4.804611176240719e-05, 'epoch': 0.12}
{'loss': 0.6499, 'learning_rate': 4.789581266720775e-05, 'epoc

The following columns in the evaluation set don't have a corresponding argument in `RobertaForSequenceClassification.forward` and have been ignored: tokens. If tokens are not expected by `RobertaForSequenceClassification.forward`,  you can safely ignore this message.
***** Running Evaluation *****
  Num examples = 98568
  Batch size = 32


...

Saving model checkpoint to ./results/TempRelTest\checkpoint-30000
Configuration saved in ./results/TempRelTest\checkpoint-30000\config.json


{'eval_loss': 0.7842873930931091, 'eval_precision': 0.7817344371398426, 'eval_recall': 0.7817344371398426, 'eval_f1': 0.7817344371398426, 'eval_runtime': 1939.6504, 'eval_samples_per_second': 50.817, 'eval_steps_per_second': 1.588, 'epoch': 0.54}


Model weights saved in ./results/TempRelTest\checkpoint-30000\pytorch_model.bin
tokenizer config file saved in ./results/TempRelTest\checkpoint-30000\tokenizer_config.json
Special tokens file saved in ./results/TempRelTest\checkpoint-30000\special_tokens_map.json


{'loss': 0.783, 'learning_rate': 4.083175519283374e-05, 'epoch': 0.55}
{'loss': 0.7756, 'learning_rate': 4.0681456097634296e-05, 'epoch': 0.56}
{'loss': 0.7763, 'learning_rate': 4.053115700243485e-05, 'epoch': 0.57}
{'loss': 0.7827, 'learning_rate': 4.0380857907235394e-05, 'epoch': 0.58}
{'loss': 0.7895, 'learning_rate': 4.023055881203595e-05, 'epoch': 0.59}
{'loss': 0.7966, 'learning_rate': 4.008025971683651e-05, 'epoch': 0.6}
{'loss': 0.771, 'learning_rate': 3.9929960621637056e-05, 'epoch': 0.6}
{'loss': 0.7691, 'learning_rate': 3.977966152643761e-05, 'epoch': 0.61}
{'loss': 0.7899, 'learning_rate': 3.962936243123817e-05, 'epoch': 0.62}
{'loss': 0.791, 'learning_rate': 3.947906333603872e-05, 'epoch': 0.63}
{'loss': 0.7927, 'learning_rate': 3.9328764240839273e-05, 'epoch': 0.64}
{'loss': 0.7759, 'learning_rate': 3.917846514563982e-05, 'epoch': 0.65}
{'loss': 0.7739, 'learning_rate': 3.902816605044038e-05, 'epoch': 0.66}
{'loss': 0.7994, 'learning_rate': 3.8877866955240935e-05, 'epoch'

KeyboardInterrupt: 

In [12]:
temprel_trainer.save_model("./results/TempRelTest")
temprel_trainer.tokenizer.save_pretrained("./results/TempRelTest")

Saving model checkpoint to ./results/TempRelTest
Configuration saved in ./results/TempRelTest\config.json
Model weights saved in ./results/TempRelTest\pytorch_model.bin
tokenizer config file saved in ./results/TempRelTest\tokenizer_config.json
Special tokens file saved in ./results/TempRelTest\special_tokens_map.json
tokenizer config file saved in ./results/TempRelTest\tokenizer_config.json
Special tokens file saved in ./results/TempRelTest\special_tokens_map.json


('./results/TempRelTest\\tokenizer_config.json',
 './results/TempRelTest\\special_tokens_map.json',
 './results/TempRelTest\\vocab.json',
 './results/TempRelTest\\merges.txt',
 './results/TempRelTest\\added_tokens.json',
 './results/TempRelTest\\tokenizer.json')

In [ ]:
temprel_trainer.evaluate(datasets["test"])

In [17]:
preds = temprel_trainer.predict(datasets["test"])

The following columns in the test set don't have a corresponding argument in `RobertaForSequenceClassification.forward` and have been ignored: tokens. If tokens are not expected by `RobertaForSequenceClassification.forward`,  you can safely ignore this message.
***** Running Prediction *****
  Num examples = 246420
  Batch size = 32


In [36]:
outputs = np.argmax(preds[0], axis=1)

In [35]:
actuals = np.array(list(datasets["test"]["labels"]))

In [39]:
len(actuals[outputs == actuals])/len(outputs)

0.7817303790276763

In [40]:
np.unique(outputs, return_counts=True)

(array([1], dtype=int64), array([246420], dtype=int64))

In [41]:
preds[0][0]

array([ 0.1440946 ,  3.3807616 ,  1.5376853 , -0.61053634, -0.6982857 ,
       -4.098998  , -1.5623726 , -3.7268841 ], dtype=float32)

In [42]:
label_list

['AFTER',
 'BEFORE',
 'CONTAINS',
 'DURING',
 'EQUALS',
 'FINISHES',
 'OVERLAPS',
 'STARTS']